# 3.2.3 — Projeção combinada de **Tokens + Documentos** com BERT

**Aluna:** Bruna Soto Cardoso dos Santos  
**Corpus:** 35 documentos — Ciência e Tecnologia no Brasil (AD1)  
**Repositório:** https://github.com/brusoto/SRI_AD_TE

---
Projeta **tokens e documentos no mesmo espaço vetorial** BERT.  
Permite ver quais tokens são mais representativos de cada documento.

**Saídas** (salvas em `projecao/`):
- `tensors_token_documento.tsv`
- `metadata_token_documento.tsv`
- `config_token_documento.json`

## Célula 1 — Instalação

In [ ]:
!pip install transformers torch --quiet

## Célula 2 — Drive e caminhos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE_PATH = '/content/drive/MyDrive/Colab Notebooks/SRI'
DATA_PATH = os.path.join(BASE_PATH, 'data')
PROJ_PATH = os.path.join(BASE_PATH, 'projecao')
os.makedirs(PROJ_PATH, exist_ok=True)
CSV_FILE  = os.path.join(DATA_PATH, 'documentos.csv')
print(f'CSV: {CSV_FILE}')

## Célula 3 — Baixar corpus se necessário

In [ ]:
if not os.path.exists(CSV_FILE):
    os.makedirs(DATA_PATH, exist_ok=True)
    URL = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/data/documentos.csv'
    !wget -q "{URL}" -O "{CSV_FILE}"
    print('Baixado.')
else:
    print('Arquivo já existe.')

## Célula 4 — Imports e corpus

In [ ]:
import pandas as pd
import torch
import numpy as np
import string
from transformers import BertTokenizer, BertModel

df = pd.read_csv(CSV_FILE)
df.columns = [c.strip().lower() for c in df.columns]
if 'id' not in df.columns:
    df.insert(0, 'id', range(1, len(df)+1))

print(f'Documentos: {len(df)}')

## Célula 5 — Modelo BERT

In [ ]:
MODEL_NAME = 'bert-base-multilingual-cased'
tokenizer  = BertTokenizer.from_pretrained(MODEL_NAME)
model      = BertModel.from_pretrained(MODEL_NAME)
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model  = model.to(device)
print(f'Dispositivo: {device}')

## Célula 6 — Parâmetros

In [ ]:
# Tokens por documento incluídos na projeção combinada
MAX_TOKENS_POR_DOC = 10  # menos que no notebook 3.2.2 para não poluir o gráfico
PONTUACOES = set(string.punctuation)

## Célula 7 — Geração dos embeddings (tokens + documentos)

In [ ]:
all_embeddings = []   # vetor 768-dim
all_metadata   = []   # (label, tipo, doc_id)

for _, row in df.iterrows():
    doc_id = row['id']
    texto  = str(row['documento'])

    encoded = tokenizer(
        texto,
        return_tensors='pt',
        max_length=512,
        truncation=True,
        padding='max_length'
    )
    encoded_dev = {k: v.to(device) for k, v in encoded.items()}

    with torch.no_grad():
        outputs = model(**encoded_dev)

    hidden = outputs.last_hidden_state.squeeze(0).cpu().numpy()
    ids    = encoded['input_ids'].squeeze().tolist()
    tokens = tokenizer.convert_ids_to_tokens(ids)

    # ── 1. Embedding do DOCUMENTO (token [CLS]) ───────────────────────────
    cls_emb = hidden[0]
    resumo  = ' '.join(texto.split()[:5])
    all_embeddings.append(cls_emb)
    all_metadata.append((f'DOC_{doc_id}: {resumo}', 'DOCUMENTO', f'Doc_{doc_id}'))

    # ── 2. Embeddings dos TOKENS do documento ────────────────────────────
    count = 0
    for idx, (tok, emb) in enumerate(zip(tokens, hidden)):
        if tok in ('[CLS]', '[SEP]', '[PAD]'):
            continue
        if tok.startswith('##'):
            continue
        if tok in PONTUACOES:
            continue
        if count >= MAX_TOKENS_POR_DOC:
            break

        all_embeddings.append(emb)
        all_metadata.append((f'{tok} [Doc_{doc_id}]', 'TOKEN', f'Doc_{doc_id}'))
        count += 1

    print(f'Doc_{doc_id:02d}: 1 doc + {count} tokens')

all_embeddings_np = np.array(all_embeddings)
print(f'\nTotal pontos: {len(all_metadata)} | Shape: {all_embeddings_np.shape}')

## Célula 8 — Salvar arquivos TSV

In [ ]:
# tensors
tensor_path = os.path.join(PROJ_PATH, 'tensors_token_documento.tsv')
np.savetxt(tensor_path, all_embeddings_np, delimiter='\t')
print(f'Salvo: {tensor_path}')

# metadata
meta_path = os.path.join(PROJ_PATH, 'metadata_token_documento.tsv')
with open(meta_path, 'w', encoding='utf-8') as f:
    f.write('label\ttipo\tdoc\n')
    for label, tipo, doc in all_metadata:
        f.write(f'{label}\t{tipo}\t{doc}\n')
print(f'Salvo: {meta_path}')

## Célula 9 — Gerar config_token_documento.json

In [ ]:
import json

GITHUB_RAW = 'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao'
config = {
    "embeddings": [
        {
            "tensorName": "Tokens e Documentos — Ciência e Tecnologia BR",
            "tensorShape": [len(all_metadata), 768],
            "tensorPath": f"{GITHUB_RAW}/tensors_token_documento.tsv",
            "metadataPath": f"{GITHUB_RAW}/metadata_token_documento.tsv"
        }
    ]
}

config_path = os.path.join(PROJ_PATH, 'config_token_documento.json')
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
print(f'Salvo: {config_path}')

## Célula 10 — Visualização PCA com distinção DOCUMENTO vs TOKEN

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

pca    = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(all_embeddings_np)

fig, ax = plt.subplots(figsize=(15, 10))

for i, (x, y) in enumerate(coords):
    label, tipo, doc = all_metadata[i]

    if tipo == 'DOCUMENTO':
        ax.scatter(x, y, color='red', s=120, marker='*', zorder=5, edgecolors='darkred', linewidths=0.5)
        ax.annotate(label.split(':')[0],  # só 'DOC_X'
                    (x, y), fontsize=7, fontweight='bold', color='darkred',
                    xytext=(3, 3), textcoords='offset points')
    else:
        ax.scatter(x, y, color='steelblue', s=25, alpha=0.6, zorder=3)
        if i % 20 == 0:
            tok = label.split(' [')[0]
            ax.annotate(tok, (x, y), fontsize=6, alpha=0.7,
                        xytext=(2, 2), textcoords='offset points')

legend = [
    mpatches.Patch(color='red',      label='Documentos (embedding [CLS])'),
    mpatches.Patch(color='steelblue', label='Tokens (embedding contextual)'),
]
ax.legend(handles=legend, loc='upper right', fontsize=9)
ax.set_title('Projeção PCA — Tokens e Documentos no mesmo espaço (BERT)\nCorpus: Ciência e Tecnologia no Brasil',
             fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.grid(True, alpha=0.3)
plt.tight_layout()

fig_path = os.path.join(PROJ_PATH, 'grafico_token_documento.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Salvo: {fig_path}')

## Célula 11 — Link para o TensorFlow Embedding Projector

In [ ]:
PROJECTOR_URL = (
    'https://projector.tensorflow.org/?config='
    'https://raw.githubusercontent.com/brusoto/SRI_AD_TE/main/projecao/config_token_documento.json'
)
print('═' * 70)
print('LINK DO EMBEDDING PROJECTOR — TOKENS + DOCUMENTOS:')
print(PROJECTOR_URL)
print('═' * 70)